# Data Inventory

Purpose: inspect the raw HALO event data before making feature definitions or claims.

This notebook answers:

- What files are available?
- What columns exist?
- How many rows and games are present?
- What event types exist?
- Are shots, goals, deflections, and follow-up events represented clearly?
- Are coordinates and xG fields complete enough for the outside-shot-value project?

In [2]:
from pathlib import Path

import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
HALO_RAW = DATA_RAW / "halo_2026"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)
print("HALO folder:", HALO_RAW)
print("HALO folder exists:", HALO_RAW.exists())

Project root: c:\Users\rinal\OneDrive\Documents\hockey-analytics\outside-shot-value
Raw data folder: c:\Users\rinal\OneDrive\Documents\hockey-analytics\outside-shot-value\data\raw
HALO folder: c:\Users\rinal\OneDrive\Documents\hockey-analytics\outside-shot-value\data\raw\halo_2026
HALO folder exists: True


In [4]:
raw_files = sorted(HALO_RAW.glob("*"))

for file in raw_files:
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file.name}: {size_mb:.2f} MB")


events.parquet: 33.99 MB
games.parquet: 0.03 MB
players.parquet: 0.09 MB
stints.parquet: 9.26 MB
tracking.parquet: 164.01 MB


In [5]:
parquet_files = sorted(HALO_RAW.glob("*.parquet"))

parquet_files

[WindowsPath('c:/Users/rinal/OneDrive/Documents/hockey-analytics/outside-shot-value/data/raw/halo_2026/events.parquet'),
 WindowsPath('c:/Users/rinal/OneDrive/Documents/hockey-analytics/outside-shot-value/data/raw/halo_2026/games.parquet'),
 WindowsPath('c:/Users/rinal/OneDrive/Documents/hockey-analytics/outside-shot-value/data/raw/halo_2026/players.parquet'),
 WindowsPath('c:/Users/rinal/OneDrive/Documents/hockey-analytics/outside-shot-value/data/raw/halo_2026/stints.parquet'),
 WindowsPath('c:/Users/rinal/OneDrive/Documents/hockey-analytics/outside-shot-value/data/raw/halo_2026/tracking.parquet')]

In [6]:
events_path = HALO_RAW / "events.parquet"

events = pd.read_parquet(events_path)

events.shape

(1800464, 24)

## Dataset Shape and Columns

Before defining shots, rebounds, slot occupancy, or possession outcomes, we need to inspect the basic structure of each raw table.

This section checks:

- row and column counts
- column names
- data types
- missingness
- whether the key event table matches the expected HALO data structure

In [7]:
# Load the smaller metadata tables so we can inspect the full raw data package.
# We already loaded events above because it is the central event log.

games = pd.read_parquet(HALO_RAW / "games.parquet")
players = pd.read_parquet(HALO_RAW / "players.parquet")
stints = pd.read_parquet(HALO_RAW / "stints.parquet")

# Tracking is larger, but still manageable. We load it now because slot occupancy
# depends on player locations at the moment of the event.
tracking = pd.read_parquet(HALO_RAW / "tracking.parquet")

tables = {
    "events": events,
    "games": games,
    "players": players,
    "stints": stints,
    "tracking": tracking,
}

summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
        }
        for name, df in tables.items()
    ]
)

summary

,table,rows,columns
0,events,1800464,24
1,games,480,12
2,players,1172,8
3,stints,2212064,15
4,tracking,13529224,10


In [8]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())


EVENTS
['game_id', 'period', 'period_time', 'game_stint', 'sl_event_id', 'sequence_id', 'player_id', 'player_name', 'team', 'team_id', 'opp_team', 'opp_team_id', 'event_type', 'outcome', 'flags', 'description', 'detail', 'sl_xg_all_shots', 'x', 'y', 'x_adj', 'y_adj', 'has_tracking_data', 'event_player_tracked']

GAMES
['game_id', 'game_date', 'league', 'season', 'home_team', 'away_team', 'home_team_id', 'away_team_id', 'home_score', 'away_score', 'game_outcome', 'home_start_net']

PLAYERS
['player_id', 'player_name', 'last_name', 'first_name', 'handed', 'birth_date', 'position_group', 'primary_position']

STINTS
['game_id', 'period', 'period_time_start', 'period_time_end', 'game_stint', 'n_home_skaters', 'n_away_skaters', 'is_home_net_empty', 'is_away_net_empty', 'home_score', 'away_score', 'player_id', 'player_name', 'team_id', 'team']

TRACKING
['game_id', 'sl_event_id', 'team_id', 'team_name', 'player_id', 'player_name', 'tracking_x', 'tracking_y', 'tracking_vel_x', 'tracking_vel_y

In [9]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    display(df.head())


EVENTS


,game_id,period,period_time,game_stint,sl_event_id,sequence_id,player_id,player_name,team,team_id,...,flags,description,detail,sl_xg_all_shots,x,y,x_adj,y_adj,has_tracking_data,event_player_tracked
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,0,1,NaN,NaN,NaN,NaN,...,NaN,Face-Off,nz,NaN,NaN,NaN,NaN,NaN,1,0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,1,1,8cdcb61e-d733-bde6-a101-ef3140e48149,"L'Esperance, Joel",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,...,"centerfodot, righthandedopponent",NZ FACE OFF-,none,NaN,-0.201431,-0.755550,-0.201431,-0.755550,1,1
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,2,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,"centerfodot, righthandedopponent",NZ REC FACE OFF+ENTRY,recoveredwithentry,NaN,-0.201431,-0.755550,0.201431,0.755550,1,1
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.070000000,1.0,3,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,NaN,F/OFF LPR+ NZ,faceoff,NaN,0.807243,1.257381,-0.807243,-1.257381,1,1
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.130000000,1.0,4,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,eastside,NZPASS south+,south,NaN,0.304306,0.845894,-0.304306,-0.845894,1,1



GAMES


,game_id,game_date,league,season,home_team,away_team,home_team_id,away_team_id,home_score,away_score,game_outcome,home_start_net
0,0f51a5c7-6d54-1917-d4a5-9e83a0b4a1ea,2024-04-20,AHL,2023,CHI,IA,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,50030ea0-2c9b-66e2-a36c-bab16b79492c,2,3,away_win,pos_x
1,0f1f9e60-c5cc-30e1-f520-e92f2da2ae08,2024-04-20,AHL,2023,BAK,HEN,19188913-73c8-34a7-d02a-5d1c3e58a191,411d86dc-8d7b-5507-c28d-87aa160b61cc,5,3,home_win,neg_x
2,1a0aceea-b3da-60e0-4c5c-0ba532fa6731,2024-04-20,AHL,2023,ABB,CAL,971502f4-80be-6e90-0bfb-3a60367a24ff,fbd255f8-427d-f29b-e83e-6c631c75899a,3,2,home_win,pos_x
3,609644d8-b5b8-8973-339a-53ca2c0d4ce1,2024-04-20,AHL,2023,HER,CLT,c1cd64c9-e2fd-7a1a-269a-aa5a8e422cbb,2251a5f6-9710-c610-8d44-9f2e4f4fc7a2,1,4,away_win,neg_x
4,2730a3bf-1f50-b700-7bd4-f01369b5e5cf,2024-04-20,AHL,2023,TEX,MB,9bfe142b-8c31-828f-dde8-a252d75d5ef3,1f9a39ec-18eb-166c-d1ed-5dfb9343f764,1,4,away_win,neg_x



PLAYERS


,player_id,player_name,last_name,first_name,handed,birth_date,position_group,primary_position
0,58ff86bf-e659-f9dc-3f5f-6a51741e47ab,"Zohorna, Radim",Zohorna,Radim,L,1996-04-29,F,C
1,87bd2512-ca41-d650-0517-297bbfc26a41,"Philp, Luke",Philp,Luke,R,1995-11-06,F,C
2,cb889642-eedd-496a-9c02-563f569c6359,"Roos, Filip",Roos,Filip,L,1999-01-05,D,D
3,69c1e284-b220-e453-250a-66413a82618e,"Studenic, Marian",Studenic,Marian,L,1998-10-28,F,LW
4,f8513815-ad09-b742-fef8-9814f2313b80,"Steen, Oskar",Steen,Oskar,R,1998-03-09,F,RW



STINTS


,game_id,period,period_time_start,period_time_end,game_stint,n_home_skaters,n_away_skaters,is_home_net_empty,is_away_net_empty,home_score,away_score,player_id,player_name,team_id,team
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,31.000000000,1,5,5,False,False,0,0,9cdb062f-6fc7-e205-1858-d4f6f32237a4,"Johansson, Albert",6cac12e2-0546-2c1a-689f-ab26d8a6355a,GR
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,31.000000000,1,5,5,False,False,0,0,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,CLE
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,31.000000000,1,5,5,False,False,0,0,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,CLE
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,31.000000000,1,5,5,False,False,0,0,8682f8c1-304a-51d9-6663-158ca58fa6c9,"Didier, Josiah",6cac12e2-0546-2c1a-689f-ab26d8a6355a,GR
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,31.000000000,1,5,5,False,False,0,0,6eb5f6c3-b657-5df9-6027-43ce18a2cca6,"Greaves, Jet",d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,CLE



TRACKING


,game_id,sl_event_id,team_id,team_name,player_id,player_name,tracking_x,tracking_y,tracking_vel_x,tracking_vel_y
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,Monsters,0f052c34-6cd2-f4fd-a861-d9051a8e86e4,"Angle, Tyler",1.735564,-12.985565,NaN,NaN
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,663df049-a045-0561-16e3-1db633a0723e,"Aston-Reese, Zachary",-2.552494,13.848426,NaN,NaN
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,015f554a-21c0-99bc-0a31-a176810b40c6,"Shine, Dominik",-2.368766,-12.660762,NaN,NaN
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,Monsters,NaN,NaN,0.216535,13.635171,NaN,NaN
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,9cdb062f-6fc7-e205-1858-d4f6f32237a4,"Johansson, Albert",-20.072179,7.312992,NaN,NaN


## Initial Event Structure Checks

The event table is not a simple play-by-play feed where each hockey action maps neatly to one row.

Some hockey situations appear to generate multiple related event rows. For example, an opening faceoff can include a higher-level event row, player-specific participant rows, and a later row representing the possession outcome.

Before analyzing shots, rebounds, deflections, or follow-up value, we need to inspect how key event types are represented.

In [10]:
# Count event types to understand the vocabulary of the event table.
# This tells us which hockey actions are represented and how frequent they are.

event_type_counts = (
    events["event_type"]
    .value_counts(dropna=False)
    .rename_axis("event_type")
    .reset_index(name="rows")
)

event_type_counts

,event_type,rows
0,pass,402670
1,lpr,338836
2,reception,304903
3,carry,110362
4,failedpasslocation,97366
5,faceoff,84621
6,block,66233
7,puckprotection,54827
8,shot,53971
9,pressure,53961


In [11]:
# Inspect the first sequence of the first game.
# Because sequence_id is defined as a faceoff-to-whistle stretch,
# this helps us understand how faceoffs and immediate possession outcomes are encoded.

first_game_id = events["game_id"].iloc[0]
first_sequence_id = events.loc[events["game_id"] == first_game_id, "sequence_id"].iloc[0]

first_sequence = events[
    (events["game_id"] == first_game_id)
    & (events["sequence_id"] == first_sequence_id)
].sort_values(["period", "period_time", "sl_event_id"])

first_sequence.head(25)

,game_id,period,period_time,game_stint,sl_event_id,sequence_id,player_id,player_name,team,team_id,...,flags,description,detail,sl_xg_all_shots,x,y,x_adj,y_adj,has_tracking_data,event_player_tracked
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,0,1,NaN,NaN,NaN,NaN,...,NaN,Face-Off,nz,NaN,NaN,NaN,NaN,NaN,1,0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,1,1,8cdcb61e-d733-bde6-a101-ef3140e48149,"L'Esperance, Joel",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,...,"centerfodot, righthandedopponent",NZ FACE OFF-,none,NaN,-0.201431,-0.755550,-0.201431,-0.755550,1,1
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,2,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,"centerfodot, righthandedopponent",NZ REC FACE OFF+ENTRY,recoveredwithentry,NaN,-0.201431,-0.755550,0.201431,0.755550,1,1
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.070000000,1.0,3,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,NaN,F/OFF LPR+ NZ,faceoff,NaN,0.807243,1.257381,-0.807243,-1.257381,1,1
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.130000000,1.0,4,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,eastside,NZPASS south+,south,NaN,0.304306,0.845894,-0.304306,-0.845894,1,1
5,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2.200000000,1.0,5,1,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,NaN,N-ZONE PASS RECEPTION,regular,NaN,6.339600,20.917810,-6.339600,-20.917810,1,1
6,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2.470000000,1.0,6,1,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,eastside,NZPASS north+,north,NaN,6.339600,23.386793,-6.339600,-23.386793,1,1
7,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,3.370000000,1.0,7,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,NaN,N-ZONE PASS RECEPTION,regular,NaN,-23.839119,20.872547,23.839119,-20.872547,1,1
8,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,4.430000000,1.0,8,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,eastwest,NORTH CYCLE OFFBOARDS-,northoffboards,NaN,-43.451580,25.992952,43.451580,-25.992952,1,1
9,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,7.130000000,1.0,9,1,0f052c34-6cd2-f4fd-a861-d9051a8e86e4,"Angle, Tyler",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,NaN,MISS PASS,regular,NaN,-92.236880,-20.369087,92.236880,20.369087,0,0


In [12]:
first_sequence.shape

(77, 24)

In [13]:
first_sequence["event_type"].value_counts(dropna=False) 

event_type
pass                      20
lpr                       15
reception                 15
failedpasslocation         5
faceoff                    3
dumpout                    3
dumpin                     3
dumpinagainst              3
carry                      3
puckprotection             2
controlledbreakout         1
block                      1
controlledentryagainst     1
icing                      1
whistle                    1
Name: count, dtype: int64

In [14]:
first_sequence[
    [
        "period",
        "period_time",
        "sl_event_id",
        "player_name",
        "team",
        "event_type",
        "outcome",
        "description",
        "detail",
        "x_adj",
        "y_adj",
    ]
].tail(20)

,period,period_time,sl_event_id,player_name,team,event_type,outcome,description,detail,x_adj,y_adj
57,1,68.730000000,57,"Clayton, Cole",CLE,dumpout,failed,OFF GLASS DUMP OUT-,boards,-90.833725,-24.392675
58,1,69.770000000,58,"Edvinsson, Simon",GR,lpr,successful,LPR+ OZ,error,36.013123,34.451496
59,1,70.800000000,59,"Edvinsson, Simon",GR,pass,successful,NORTH CYCLE +,north,34.001358,31.982515
60,1,71.300000000,60,"Mazur, Carter",GR,reception,successful,O-ZONE PASS RECEPTION,regular,41.545486,36.006046
61,1,72.130000000,61,"Mazur, Carter",GR,puckprotection,failed,OZDEKE-,deke,40.539597,30.519423
62,1,75.800000000,62,"Christiansen, Jake",CLE,lpr,successful,LPR+ DZ,error,-79.266070,-1.303104
63,1,75.970000000,63,"Christiansen, Jake",CLE,dumpout,successful,DUMP OUT+,ice,-81.780780,-1.257381
64,1,81.330000000,64,"Tuomisto, Antti",GR,lpr,successful,DUMP IN LPR+,opdump,-49.989815,-16.848503
65,1,81.930000000,65,"Tuomisto, Antti",GR,pass,successful,DZ OUTLET PASS+,outlet,-47.475110,-30.336456
66,1,84.830000000,66,"Berggren, Jonatan",GR,reception,successful,N-ZONE PASS RECEPTION,regular,-4.725105,8.298557


In [15]:
# Look specifically at shot and deflection rows.
# We need to know whether shots, deflections, and xG are represented as separate events,
# because that affects how we measure follow-up value.

shot_like_events = events[
    events["event_type"].isin(["shot", "deflection", "goal"])
].copy()

shot_like_events[
    [
        "game_id",
        "period",
        "period_time",
        "sl_event_id",
        "sequence_id",
        "player_name",
        "team",
        "event_type",
        "outcome",
        "detail",
        "sl_xg_all_shots",
        "x_adj",
        "y_adj",
        "has_tracking_data",
        "event_player_tracked",
    ]
].head(30)

,game_id,period,period_time,sl_event_id,sequence_id,player_name,team,event_type,outcome,detail,sl_xg_all_shots,x_adj,y_adj,has_tracking_data,event_player_tracked
102,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,125.270000000,102,2,"Fix-Wolansky, Trey",CLE,shot,successful,slot,0.106327,67.089810,-1.760323,1,1
111,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,142.200000000,111,2,"Sweezey, Billy",CLE,shot,failed,outside,0.002135,37.919224,-26.907381,1,1
122,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,155.270000000,122,2,"Sillinger, Owen",CLE,shot,successful,outside,0.018751,79.663345,19.866150,1,0
129,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,161.830000000,129,2,"Fix-Wolansky, Trey",CLE,shot,successful,outside,0.016561,79.663345,-25.490011,1,1
138,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,174.970000000,138,2,"Sillinger, Owen",CLE,shot,failed,outsideblocked,0.002630,35.404520,-23.935457,1,0
151,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,190.000000000,151,2,"Kasper, Marco",GR,shot,successful,outside,0.003101,41.042540,31.433851,1,1
174,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,211.370000000,174,2,"Didier, Josiah",GR,shot,successful,outside,0.006018,61.160187,-29.879250,1,1
280,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,320.500000000,280,4,"Jiricek, David",CLE,shot,failed,outsideblocked,0.018497,50.994990,17.852940,1,1
292,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,331.100000000,292,4,"Jiricek, David",CLE,shot,successful,slot,0.051982,69.603810,21.876472,1,1
293,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,331.500000000,293,4,"Jiricek, David",CLE,goal,successful,none,NaN,69.603810,21.876472,1,1


## Inventory Findings

- HALO data contains five parquet files: events, games, players, stints, and tracking.
- The event table has 1,800,464 rows and 24 columns.
- Tracking data is available separately at one row per player per event.
- Shot, deflection, block, assist, goal, and pressure events require event-grammar inspection before building a shot-value dataset.
- Detailed shot, goal, block, and deflection encoding is handled separately in `02_event_grammar.ipynb`.